# De una reseña a un embedding

En VOC, los comentarios preparados todavía son texto. Para comparar su contenido y después buscar agrupamientos, necesitas representarlos mediante vectores numéricos. Un **embedding** conserva información útil para esas comparaciones, aunque no explica por sí solo un problema ni asigna una etiqueta de sentimiento.

En esta libreta convertirás seis frases inventadas en vectores. Primero observarás los tokens, después abrirás el cálculo del encoder y finalmente compararás similitudes. Al terminar, conectarás este ejemplo con los embeddings del corpus de VOC que ya se produjeron y verificaron.

## Para comenzar

Abre esta libreta desde `just jupyter-reports`, en `embeddings/`, con el kernel del **Operador**. Necesitas las dependencias de VOC y la revisión de MiniLM ya descargada en `data/models/sentence-transformers`.

El recorrido ejecuta inferencia local sobre seis frases, sin entrenar un modelo, consultar ZenML, escribir en la caché de vectores ni recalcular el corpus. No descarga modelos. Si los archivos no están disponibles, detente y prepara el modelo mediante el procedimiento del Operador.

Ejecuta las celdas en orden. Los datos del ejemplo son inventados y sus similitudes no constituyen una evaluación del corpus real.

## 1. Preparando frases que puedas reconocer

Ejecuta esta celda. Incluye dos formas de describir un problema de acceso, una experiencia positiva de acceso, otros asuntos y una repetición deliberada.

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Localizar el checkout tanto desde reports como desde notebooks/reports.
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src/voc").is_dir() and (path / "pyproject.toml").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Abre la libreta desde el checkout de VOC.")
os.environ.setdefault("VOC_DATA_DIR", str(ROOT / "data"))

reviews = pd.DataFrame({
    "record_id": ["ejemplo-1", "ejemplo-2", "ejemplo-3", "ejemplo-4", "ejemplo-5", "ejemplo-6"],
    "model_text": [
        "No puedo iniciar sesión en la aplicación.",
        "La app no me deja entrar a mi cuenta.",
        "Puedo iniciar sesión sin problemas.",
        "Mi transferencia no llega a la cuenta de destino.",
        "Me gustaría cambiar el color de la aplicación.",
        "No puedo iniciar sesión en la aplicación.",
    ],
})
display(reviews)

EXPORT_RESULTS = True  # Exportar figuras al terminar la libreta.


,record_id,model_text
0,ejemplo-1,No puedo iniciar sesión en la aplicación.
1,ejemplo-2,La app no me deja entrar a mi cuenta.
2,ejemplo-3,Puedo iniciar sesión sin problemas.
3,ejemplo-4,Mi transferencia no llega a la cuenta de destino.
4,ejemplo-5,Me gustaría cambiar el color de la aplicación.
5,ejemplo-6,No puedo iniciar sesión en la aplicación.


Las filas 1 y 6 tienen identificadores diferentes aunque comparten texto. Conserva esa distinción: **identidad del registro** e **igualdad del texto** resuelven problemas distintos.

Una **política de preparación** es una regla explícita y versionada que define qué texto recibe el encoder. No es un modelo ni un prompt. En VOC, `verbatim-v1` significa conservar literalmente el campo `model_text` en `embedding_text`. `v1` identifica la versión de esa regla, que permite registrar y reproducir la preparación.

Por ejemplo, «No puedo iniciar sesión en la aplicación.» conserva su negación, mayúscula, acento y puntuación. Aplica la función existente y compara entrada y salida.

In [2]:
from voc_ml.embeddings.features import prepare_frame
from voc_ml.embeddings.features.contracts import FeatureConfig

features = prepare_frame(reviews, FeatureConfig(policy="verbatim-v1"))
assert features["embedding_text"].equals(reviews["model_text"])
display(features[["record_id", "model_text", "embedding_text"]])
print("Registros:", len(features))
print("Textos distintos:", features["embedding_text"].nunique())
print("Hash del primer texto:", features.loc[0, "feature_text_hash"])

,record_id,model_text,embedding_text
0,ejemplo-1,No puedo iniciar sesión en la aplicación.,No puedo iniciar sesión en la aplicación.
1,ejemplo-2,La app no me deja entrar a mi cuenta.,La app no me deja entrar a mi cuenta.
2,ejemplo-3,Puedo iniciar sesión sin problemas.,Puedo iniciar sesión sin problemas.
3,ejemplo-4,Mi transferencia no llega a la cuenta de destino.,Mi transferencia no llega a la cuenta de destino.
4,ejemplo-5,Me gustaría cambiar el color de la aplicación.,Me gustaría cambiar el color de la aplicación.
5,ejemplo-6,No puedo iniciar sesión en la aplicación.,No puedo iniciar sesión en la aplicación.


Registros: 6
Textos distintos: 5
Hash del primer texto: 01736a2dffcb2517b7a3327689a90a7490431a4a5967d838d131648dd2759bcd


**Comprobación:** obtienes seis registros y cinco textos distintos. El hash identifica el texto exacto para verificarlo o reutilizar su cálculo. No es un embedding y no mide similitud: dos paráfrasis suelen tener hashes distintos.

La preservación se refiere a esta etapa de preparación. El tokenizer del modelo aplica posteriormente su propio procesamiento.

## 2. Configurando el modelo preentrenado

Un **encoder** transforma el texto en una representación numérica. En VOC, nuestro trabajo consiste en seleccionar, configurar y ejecutar un modelo ya entrenado. El perfil `minilm-verbatim-v1` fija esa configuración y usa la política de preparación descrita arriba.

| Opción | Configuración de VOC |
| --- | --- |
| Modelo | `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` |
| Revisión | `e8f8c211226b894fcb81acc59f3b34ba3efd5f42` |
| Salida | 384 coordenadas de tipo `float32` por texto |
| Longitud máxima | 128 tokens, incluidos los especiales |
| Normalización | L2, vector de norma unitaria |
| Prompt añadido | Ninguno |
| Preparación | `verbatim-v1`, sin cambios a `model_text` |

Carga los archivos locales con la misma clase que utiliza el productor y comprueba la configuración efectiva.

In [3]:
from voc.paths import data_root
from voc.embeddings.shared.profiles import resolve_profile
from voc_ml.embeddings.encoding.sentence_encoder import SentenceEncoder

profile = resolve_profile("minilm-verbatim-v1")
encoder = SentenceEncoder(
    profile,
    data_root() / "models" / "sentence-transformers",
    local_files_only=True,
)
assert encoder.resolved_revision == profile.revision
display(pd.Series(profile.config(), name="Configuración"))

/home/node/.cache/uv/archive-v0/-_OFAkQgdQKRZ2EY/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15279.94it/s]

model                 sentence-transformers/paraphrase-multilingual-...
revision                       e8f8c211226b894fcb81acc59f3b34ba3efd5f42
dimensions                                                          384
dtype                                                               <f4
normalize                                                          True
max_length                                                          128
prompt                                                             None
preparation_policy                                          verbatim-v1
Name: Configuración, dtype: object

**Comprobación:** el perfil declara 384 dimensiones y un máximo de 128 tokens. Esta libreta realiza **inferencia**, sin actualizar los pesos del modelo.

Sentence-BERT proporciona el fundamento para representar oraciones y compararlas mediante coseno [1]. El modelo seleccionado tokeniza el texto, obtiene representaciones contextuales y las promedia para producir un vector por frase [2]. VOC solicita además normalización L2.

Las siguientes celdas abren ese mecanismo con fines didácticos. No son operaciones que debamos reimplementar para producir embeddings: el adaptador de VOC las ejecuta mediante Sentence Transformers.

## 3. Observando tokens, relleno y truncamiento

Usa el tokenizer incluido en el modelo. Un **token** puede ser una palabra, parte de una palabra o un símbolo especial. El límite de 128 cuenta tokens, incluidos los especiales, no palabras ni caracteres.

In [4]:
texts = features["embedding_text"].tolist()
tokenizer = encoder.model.tokenizer

# Mostrar los tokens que entran realmente al modelo para la primera frase.
batch = encoder.model.preprocess(texts)
first_ids = batch["input_ids"][0].tolist()
display(pd.DataFrame({
    "posición": range(len(first_ids)),
    "token": tokenizer.convert_ids_to_tokens(first_ids),
    "id": first_ids,
    "attention_mask": batch["attention_mask"][0].tolist(),
}))
print("Forma del lote (textos, posiciones):", tuple(batch["input_ids"].shape))

,posición,token,id,attention_mask
0,0,<s>,0,1
1,1,▁No,438,1
2,2,▁puedo,62040,1
3,3,▁iniciar,54061,1
4,4,▁sesión,58873,1
5,5,▁en,22,1
6,6,▁la,21,1
7,7,▁aplicación,36050,1
8,8,.,5,1
9,9,</s>,2,1


Forma del lote (textos, posiciones): (6, 12)


El **padding** iguala las longitudes dentro del lote. La máscara marca con cero esas posiciones de relleno, que no deben contribuir al promedio. Los tokens especiales que mantienen máscara uno sí participan en el pooling de este modelo.

Para distinguir padding de truncamiento, compara una frase corta con un texto artificialmente largo. Solo tokenizarás ese texto largo, sin generar su embedding.

In [5]:
long_example = "No puedo entrar a mi cuenta. " * 80
probe_texts = [texts[0], long_example]
full_tokens = tokenizer(
    probe_texts, padding=False, truncation=False,
    add_special_tokens=True, verbose=False,
)
limited_tokens = tokenizer(
    probe_texts, padding=False, truncation=True,
    max_length=profile.max_length, add_special_tokens=True,
)
lengths = pd.DataFrame({
    "ejemplo": ["frase corta", "texto largo inventado"],
    "tokens_originales": [len(ids) for ids in full_tokens["input_ids"]],
    "tokens_entregados": [len(ids) for ids in limited_tokens["input_ids"]],
})
lengths["excede_límite"] = lengths["tokens_originales"] > profile.max_length
assert lengths.loc[1, "excede_límite"]
assert lengths.loc[1, "tokens_entregados"] == profile.max_length
display(lengths)

,ejemplo,tokens_originales,tokens_entregados,excede_límite
0,frase corta,10,10,False
1,texto largo inventado,562,128,True


**Comprobación:** el texto largo llega al límite de 128 tokens. El tokenizer conserva los tokens especiales requeridos y recorta contenido para ajustarse. El embedding representará la entrada truncada, no todo el comentario. VOC conserva el texto del registro y reporta el truncamiento como diagnóstico.

## 4. Generando los vectores

Llama al adaptador de VOC para las seis frases. La función devuelve los embeddings y el conteo de tokens anterior al truncamiento.

In [6]:
vectors, token_counts = encoder.encode(texts, batch_size=6)
assert vectors.shape == (6, 384)
assert vectors.dtype == np.dtype("float32")
assert np.isfinite(vectors).all()
np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5)

print("Matriz:", vectors.shape, "Tipo:", vectors.dtype)
display(pd.DataFrame(
    vectors[:, :8],
    index=reviews["record_id"],
    columns=[f"dimensión {i}" for i in range(8)],
).round(4))

Matriz: (6, 384) Tipo: float32


,dimensión 0,dimensión 1,dimensión 2,dimensión 3,dimensión 4,dimensión 5,dimensión 6,dimensión 7
record_id,,,,,,,,
ejemplo-1,0.0024,-0.0681,0.0200,0.0260,-0.0288,0.0022,-0.0720,0.0169
ejemplo-2,-0.0126,-0.0090,0.0286,-0.0227,0.0159,-0.0745,-0.0140,-0.0121
ejemplo-3,0.0567,-0.0291,0.0001,-0.0052,-0.0323,0.0458,0.0213,0.0231
ejemplo-4,0.0530,0.0069,0.0133,-0.0590,0.0549,-0.0971,0.0123,-0.0161
ejemplo-5,-0.0341,0.0215,-0.0109,0.0113,-0.0334,0.0188,-0.0379,-0.0988
ejemplo-6,0.0024,-0.0681,0.0200,0.0260,-0.0288,0.0022,-0.0720,0.0169


Cada fila tiene 384 coordenadas. No hay una columna reservada para «transferencias» o «sentimiento negativo». La representación aprendida distribuye información entre coordenadas. Mostrar ocho dimensiones sirve para reconocer la estructura, no para interpretar una funcionalidad.

Para abrir el mecanismo, ejecuta el Transformer y calcula el mismo **mean pooling** y la misma normalización que acabas de usar. La expresión es:

$$u = \frac{\sum_i m_i h_i}{\sum_i m_i}, \qquad v = \frac{u}{\lVert u \rVert_2}$$

Aquí $h_i$ es el vector contextual de la posición $i$, $m_i$ su máscara y $v$ el embedding normalizado. La división por la norma permite comparar direcciones.

In [7]:
import torch

# Confirmar que el ejemplo manual corresponde a la arquitectura cargada.
pooling = encoder.model[1]
assert pooling.pooling_mode == "mean"

previous_threads = torch.get_num_threads()
try:
    torch.set_num_threads(4)
    with torch.inference_mode():
        output = encoder.model[0].auto_model(**batch)
        hidden = output.last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        manual = torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy()
finally:
    torch.set_num_threads(previous_threads)

print("Vectores por token:", tuple(hidden.shape))
print("Después del pooling:", tuple(pooled.shape))
print("Diferencia máxima frente a VOC:", float(np.abs(manual - vectors).max()))
np.testing.assert_allclose(manual, vectors, atol=1e-5, rtol=1e-4)

Vectores por token: (6, 12, 384)
Después del pooling: (6, 384)
Diferencia máxima frente a VOC: 6.705522537231445e-08


**Comprobación:** pasas de una matriz por texto y posición a un vector por texto. El resultado manual coincide, dentro de tolerancia numérica, con el adaptador. Esto explica el mecanismo de esta configuración, sin reemplazar el código de producción.

## 5. Comparando textos por similitud

La **similitud coseno** compara la dirección de dos vectores:

$$\operatorname{cos}(a,b)=\frac{a\cdot b}{\lVert a\rVert_2\lVert b\rVert_2}$$

Como VOC entrega vectores normalizados, basta con su producto punto. Calcula la matriz entre las seis frases y léela junto con los textos del primer paso.

In [8]:
similarity = vectors @ vectors.T
labels = reviews["record_id"].tolist()
np.testing.assert_allclose(np.diag(similarity), 1.0, atol=1e-5)
np.testing.assert_allclose(vectors[0], vectors[5], atol=1e-5)

fig_similarity, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(similarity, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(labels)), labels=labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels=labels)
for row in range(len(labels)):
    for column in range(len(labels)):
        ax.text(column, row, f"{similarity[row, column]:.2f}", ha="center", va="center",
                color="white" if abs(similarity[row, column]) > 0.65 else "black")
ax.set_title("Similitud coseno entre frases inventadas")
fig_similarity.colorbar(image, ax=ax, label="Coseno, no probabilidad")
fig_similarity.tight_layout()
plt.show()

La diagonal y la pareja repetida deben estar cerca de uno. Para las demás parejas, observa los valores obtenidos sin imponer un orden esperado como criterio de calidad.

### Encontrando los vecinos más cercanos

Toma la primera frase como consulta y ordena los demás textos por **distancia coseno**:

$$d_{\cos}(a,b)=1-\operatorname{cos}(a,b)$$

Una similitud mayor produce una distancia menor. Para vectores no nulos, la distancia va de 0 a 2. No es una distancia euclídea ni una probabilidad. Excluye la consulta y su copia exacta para que la comparación muestre textos diferentes. Aquí recorres los cuatro candidatos restantes, sin entrenar un clasificador ni construir un índice de búsqueda.

In [9]:
from textwrap import fill

query_index = 0
query_text = texts[query_index]
# Conservar un candidato por texto y excluir todas las copias de la consulta.
candidates = features.loc[features["embedding_text"].ne(query_text)].drop_duplicates("embedding_text")
positions = candidates.index.to_numpy()
cosines = np.clip(similarity[query_index, positions], -1.0, 1.0)
neighbors = pd.DataFrame({
    "record_id": candidates["record_id"].to_numpy(),
    "texto": candidates["embedding_text"].to_numpy(),
    "similitud_coseno": cosines,
    "distancia_coseno": 1.0 - cosines,
}).sort_values(["distancia_coseno", "record_id"], kind="stable").reset_index(drop=True)
neighbors.insert(0, "orden", np.arange(1, len(neighbors) + 1))
assert len(neighbors) == 4
assert neighbors["distancia_coseno"].is_monotonic_increasing
np.testing.assert_allclose(neighbors["distancia_coseno"], 1 - neighbors["similitud_coseno"])
display(neighbors.round(4))

# Mostrar un cálculo completo para el vecino más cercano, sin redondear al ordenar.
nearest_position = features.index[features["record_id"].eq(neighbors.loc[0, "record_id"])][0]
a, b = vectors[query_index], vectors[nearest_position]
dot = float(np.dot(a, b))
norm_a, norm_b = float(np.linalg.norm(a)), float(np.linalg.norm(b))
worked_cosine = dot / (norm_a * norm_b)
np.testing.assert_allclose(worked_cosine, neighbors.loc[0, "similitud_coseno"], atol=1e-5)
print("Consulta:", query_text)
print("Vecino más cercano:", neighbors.loc[0, "texto"])
print(f"coseno = {dot:.6f} / ({norm_a:.6f} × {norm_b:.6f}) = {worked_cosine:.6f}")
print(f"distancia = 1 − {worked_cosine:.6f} = {1 - worked_cosine:.6f}")

fig_neighbors, ax = plt.subplots(figsize=(11, 5.4))
y = np.arange(len(neighbors))
ax.barh(y, neighbors["distancia_coseno"], color=["#276A96"] + ["#9DB8C9"] * (len(y) - 1), height=0.55)
ax.set_yticks(y, labels=[fill(f"{row.orden}. {row.texto}", 39) for row in neighbors.itertuples()])
ax.invert_yaxis()
ax.set_xlim(0, 2)
ax.set_xticks([0, 0.5, 1, 1.5, 2])
ax.set_xlabel("Distancia coseno = 1 − similitud coseno · menor es más cercano")
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
for i, row in neighbors.iterrows():
    label = f"d = {row.distancia_coseno:.3f}  |  cos = {row.similitud_coseno:.3f}".replace(".", ",")
    ax.text(row.distancia_coseno + 0.035, i, label, va="center", fontsize=10)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
fig_neighbors.suptitle("Vecinos cercanos entre frases inventadas", fontsize=15, fontweight="bold", y=0.98)
fig_neighbors.text(0.5, 0.89, f'Consulta: «{query_text}»', ha="center", fontsize=11)
fig_neighbors.text(0.03, 0.02,
    "Se excluyen la consulta y su copia exacta. Los cuatro candidatos se ordenan por distancia.\n"
    "La cercanía refleja el modelo seleccionado, no demuestra igual sentimiento ni una causa común.", fontsize=10)
fig_neighbors.subplots_adjust(left=0.37, right=0.98, top=0.80, bottom=0.20)
plt.show()


,orden,record_id,texto,similitud_coseno,distancia_coseno
0,1,ejemplo-2,La app no me deja entrar a mi cuenta.,0.6213,0.3787
1,2,ejemplo-3,Puedo iniciar sesión sin problemas.,0.3356,0.6644
2,3,ejemplo-5,Me gustaría cambiar el color de la aplicación.,0.2241,0.7759
3,4,ejemplo-4,Mi transferencia no llega a la cuenta de destino.,0.2163,0.7837


Consulta: No puedo iniciar sesión en la aplicación.
Vecino más cercano: La app no me deja entrar a mi cuenta.
coseno = 0.621303 / (1.000000 × 1.000000) = 0.621303
distancia = 1 − 0.621303 = 0.378697


**Comprobación:** la primera barra corresponde a la menor distancia entre los cuatro candidatos. El cálculo impreso permite seguir el producto punto, las normas y la conversión de similitud a distancia.

Este orden permite recuperar textos relacionados aunque no usen las mismas palabras. Una experiencia positiva y otra negativa pueden compartir asunto y resultar cercanas. Por eso los embeddings sirven como entrada para búsqueda y agrupamiento, mientras que sentimiento, relevancia y causas requieren análisis y evaluación propios. Las frases inventadas ilustran el procedimiento, no miden su calidad sobre el corpus.

## 6. Conservando registros al reutilizar un cálculo

La reutilización responde a igualdad exacta del texto y compatibilidad del encoder, no a similitud alta. Reconstruye las seis filas a partir de sus cinco textos distintos. Este ejemplo usa memoria y no accede a la caché persistente de VOC.

In [10]:
# Simular la reutilización sin eliminar las observaciones del corpus.
first_vector_by_text = {}
for text, vector in zip(texts, vectors, strict=True):
    first_vector_by_text.setdefault(text, vector)
reconstructed = np.stack([first_vector_by_text[text] for text in texts])
np.testing.assert_allclose(reconstructed, vectors, atol=1e-5)

print("Vectores distintos por texto:", len(first_vector_by_text))
print("Filas reconstruidas:", reconstructed.shape[0])
print("Orden conservado:", labels)

Vectores distintos por texto: 5
Filas reconstruidas: 6
Orden conservado: ['ejemplo-1', 'ejemplo-2', 'ejemplo-3', 'ejemplo-4', 'ejemplo-5', 'ejemplo-6']


**Comprobación:** cinco cálculos pueden alimentar seis registros, conservando el orden. En producción, la clave incorpora además la identidad del encoder, incluidas su configuración y versiones de software. VOC comprueba integridad de los vectores antes de reutilizarlos.

Un cambio de modelo o preparación puede impedir la reutilización. Compartir texto tampoco demuestra que dos observaciones sean el mismo evento.

## 7. Conectando el ejemplo con el corpus de VOC

El productor aplica estas operaciones por lotes, registra la ejecución en ZenML y MLflow y exporta un paquete con manifiesto y correspondencia entre registros y vectores. El lector verifica su identidad e integridad antes de entregarlo al consumidor.

La siguiente tabla transcribe la ejecución completa del **3 de noviembre de 2026**, verificada en el Operador. No se recalcula ni vuelve a comprobar el paquete al ejecutar esta celda.

In [11]:
recorded = pd.DataFrame([
    ("Registros", "84.479"),
    ("Textos distintos", "59.550"),
    ("Textos calculados en esa ejecución", "2.865"),
    ("Textos reutilizados", "56.685"),
    ("Registros con truncamiento", "153 (0,181 %)"),
    ("Matriz verificada", "(84479, 384), float32"),
], columns=["Medida", "Resultado registrado"])
display(recorded)
print("Productor:", "a3f2c83d-84c3-42cb-9cbe-8586df808491")
print("Release local:", "3f2b7b4c-7e90-431b-85d9-c45629936bb0")

,Medida,Resultado registrado
0,Registros,84.479
1,Textos distintos,59.550
2,Textos calculados en esa ejecución,2.865
3,Textos reutilizados,56.685
4,Registros con truncamiento,"153 (0,181 %)"
5,Matriz verificada,"(84479, 384), float32"


Productor: a3f2c83d-84c3-42cb-9cbe-8586df808491
Release local: 3f2b7b4c-7e90-431b-85d9-c45629936bb0


Los 153 registros truncados siguen presentes. Su representación usa una entrada acotada a 128 tokens. La matriz completa acredita disponibilidad y alineación de vectores, no la calidad de los tópicos que se construyan después.

### Reto: asunto compartido y sentimiento distinto

Sustituye una de las frases inventadas por otra que describa el mismo asunto con una valoración opuesta. Vuelve a ejecutar desde el primer paso. Anota qué similitud cambia y explica por qué ese valor no permite asignar automáticamente una etiqueta de sentimiento. Mantén seis frases para conservar las comprobaciones del ejemplo y deja intacta la repetición de las filas 1 y 6.

## Puntos clave

- **Preparación:** conserva la información del texto y declara cualquier transformación.
- **Tokens:** delimitan la entrada del modelo, con un límite distinto del número de palabras.
- **Encoder:** produce representaciones contextuales sin entrenarse en esta libreta.
- **Pooling:** resume las posiciones válidas en un vector por texto.
- **Normalización:** permite calcular similitud coseno mediante producto punto.
- **Reutilización:** evita repetir cálculos compatibles y conserva las observaciones originales.
- **Validación:** distingue integridad del artefacto de calidad semántica.

Ahora puedes explicar de dónde proviene la matriz que usará el análisis de tópicos. La elección y evaluación de agrupamientos corresponde a la siguiente etapa.

## Referencias para continuar

1. Reimers, N., y Gurevych, I. (2019). [Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks](https://arxiv.org/abs/1908.10084). Fundamento de las representaciones de oraciones comparables mediante coseno.
2. Sentence Transformers. [paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). Ficha del modelo y ejemplo de pooling. VOC fija la revisión `e8f8c211226b894fcb81acc59f3b34ba3efd5f42`.
3. Alammar, J., y Grootendorst, M. (2024). *Hands-On Large Language Models*. O’Reilly. Capítulos 2 y 5: tokens, embeddings y uso de estas representaciones para agrupar textos.

## Exportando resultados para el reporte

Exportamos las figuras, la tabla de vecinos, las frases, sus vectores y la configuración para crear el reporte. Se guardan en `data/reports/embeddings/<ejecución>/` junto con un manifiesto de procedencia e integridad.


In [12]:
import hashlib
import importlib.metadata
import json
from datetime import datetime, timezone

if EXPORT_RESULTS:
    execution = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = data_root() / "reports/embeddings" / execution
    destination.mkdir(parents=True, exist_ok=False)
    figures = {"vecinos_coseno": fig_neighbors, "similitud_ejemplos": fig_similarity}
    for name, figure in figures.items():
        figure.savefig(destination / f"{name}.png", dpi=220, bbox_inches="tight", facecolor="white")
        figure.savefig(destination / f"{name}.pdf", bbox_inches="tight", facecolor="white")
    neighbors.to_csv(destination / "vecinos_coseno.csv", index=False)
    reviews.to_csv(destination / "frases_ejemplo.csv", index=False)
    np.save(destination / "vectores_ejemplo.npy", vectors, allow_pickle=False)
    (destination / "configuracion.json").write_text(
        json.dumps(profile.as_dict(), indent=2, ensure_ascii=False), encoding="utf-8"
    )
    manifest = {
        "execution_utc": execution,
        "notebook": "reports/embeddings/produccion-de-embeddings.ipynb",
        "notebook_file_sha256": hashlib.sha256(
            (ROOT / "reports/embeddings/produccion-de-embeddings.ipynb").read_bytes()
        ).hexdigest(),
        "profile": profile.as_dict(),
        "figure_context": {
            "vecinos_coseno": "Distancia coseno calculada sobre cuatro candidatos distintos para una consulta inventada. No evalúa el corpus.",
            "similitud_ejemplos": "Inferencia local sobre seis frases inventadas. No representa resultados del corpus.",
        },
        "nearest_neighbors": {
            "query_record_id": reviews.loc[query_index, "record_id"],
            "query_text": query_text,
            "candidate_count": len(neighbors),
            "selection": "Excluir textos idénticos a la consulta y deduplicar candidatos por texto.",
            "distance": "1 - cosine_similarity",
            "ordering": "distancia_coseno ascendente, record_id para desempatar",
        },
        "historical_evidence": {
            "producer_run_id": "a3f2c83d-84c3-42cb-9cbe-8586df808491",
            "embedding_release_id": "3f2b7b4c-7e90-431b-85d9-c45629936bb0",
            "manifest_sha256": "b96fc23d9f32496312d1921d8e14e77cfcb226a8819e7328f0c4bff83447fbc2",
            "rows": 84479, "unique_texts": 59550,
            "reused_unique": 56685, "encoded_unique": 2865, "truncated_rows": 153,
        },
        "versions": {name: importlib.metadata.version(name)
                     for name in ("numpy", "matplotlib", "torch", "sentence-transformers")},
        "files_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                         for p in sorted(destination.iterdir()) if p.is_file()},
    }
    (destination / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    print("Resultados guardados en:", destination)
else:
    print("Exportación deshabilitada. Las figuras permanecen en memoria.")

Resultados guardados en: /Users/ness/workspace/voc-main/data/reports/embeddings/20261103T153947608325Z
